Ingeniería en Ciencias de la Computación y TI 

Inteligencia Artificial

Laboratorio 9

# Task 2 – Inferencia en la Red Bayesiana de Robo, Terremoto y Alarma

- Vianka Vanessa Castro Ordoñez - 23201
- Ricardo Arturo Godínez Sánchez - 23247

**Red Bayesiana** con las variables:
- **B** = Robo
- **E** = Terremoto
- **A** = Alarma


## 1. Modelo probabilístico

Las variables del problema son binarias, es decir, solo pueden valer **0** o **1**:
- `B = 1` significa que sí hubo robo.
- `E = 1` significa que sí hubo terremoto.
- `A = 1` significa que la alarma sonó.

El laboratorio define:

asumiendo que  𝜖 = 0.01 (evento raro)

\[
P(B=1)= 𝜖 = 0.01
\]

\[
P(E=1)= 𝜖 = 0.01
\]

La alarma es **determinista** y sigue una compuerta lógica OR:

La alarma solo suena ssi hay un robo o un terremoto. Es decir:

𝑃(𝐴 = 1 ∣ 𝐵, 𝐸) = [𝐴 = (𝐵 ∨ 𝐸)].
Ssi 𝐵 = 1 o 𝐸 = 1, 𝑃(𝐴 = 1) = 1. 
De lo contrario 𝑃(𝐴 = 1) = 0


Eso significa:
- Si hay robo o terremoto, entonces la alarma suena con probabilidad 1.
- Si no hay ni robo ni terremoto, la alarma no suena.


In [19]:
# Con esto ya definimos parámetros clave 
epsilon = 0.01

print(f"El valor de epsilon es: {epsilon}")

El valor de epsilon es: 0.01


## 2. Probabilidades base

Ahora vamos a crear funciones pequeñas que van a represetar las probabilidades base del problema, es decir, las que ya conocemos:

1. La probabilidad de `B`.
2. La probabilidad de `E`.
3. La probabilidad condicional `P(A | B, E)`.

En `P(A | B, E)` aplicamos la regla lógica del OR.


In [ ]:
# PROBABILIDAD DE B

def prob_b(b):
    if b == 1:
        return epsilon
    elif b == 0:
        return 1 - epsilon
    else:
        raise ValueError("El valor de B debe ser 0 o 1")
    
# PROBABILIDAD DE E
def prob_e(e):
    if e == 1:
        return epsilon
    elif e == 0:
        return 1 - epsilon
    else:
        raise ValueError("El valor de E debe ser 0 o 1")
    
# PROBABILIDAD DE A DADO B Y E
def prob_a_given_b_e(a, b, e):
    if a == 1:
        return 1 if (b == 1 or e == 1) else 0
    elif a == 0:
        return 0 if (b == 1 or e == 1) else 1
    else:
        raise ValueError("El valor de A debe ser 0 o 1")

## 3. Task 2.1 - Función de probabilidad conjunta

Escribir una función en Python prob_conjunta(b, e, a) que reciba el estado de las tres variables y retorne su probabilidad conjunta

𝑃(𝐵 = 𝑏, 𝐸 = 𝑒, 𝐴 = 𝑎)

Aplicar la regla de la cadena para Redes Bayesiana:
𝑃(𝐵, 𝐸, 𝐴) = 𝑃(𝐵) ⋅ 𝑃(𝐸) ⋅ 𝑃(𝐴 ∣ 𝐵, 𝐸)

In [21]:
def prob_conjunta(b, e, a):
    """
    Calcula la probabilidad conjunta P(B, E, A) utilizando la regla de la cadena para Redes Bayesiana:
    P(B, E, A) = P(B) * P(E) * P(A | B, E)
    """
    return prob_b(b) * prob_e(e) * prob_a_given_b_e(a, b, e)


In [22]:
## ahora realizamos las pruebas de 8 combinaciones posibles de las 3 
# variables binarias (B, E, A):
total = 0 
for b in [0, 1]:
    for e in [0, 1]:
        for a in [0, 1]:
            p = prob_conjunta(b, e, a)
            print(f"P(B={b}, E={e}, A={a}) = {p}")
            total += p
print(f"Suma total de probabilidades conjuntas: {total}")

P(B=0, E=0, A=0) = 0.9801
P(B=0, E=0, A=1) = 0.0
P(B=0, E=1, A=0) = 0.0
P(B=0, E=1, A=1) = 0.0099
P(B=1, E=0, A=0) = 0.0
P(B=1, E=0, A=1) = 0.0099
P(B=1, E=1, A=0) = 0.0
P(B=1, E=1, A=1) = 0.0001
Suma total de probabilidades conjuntas: 1.0


## 4. Task 2.2 - Inferencia Marginal

Ahora con respecto a la *marginalización* es de sumar sobre las variables ocultas:

Si quisieramos calcular:
P(A=1)

hay que sumar sobre todas las compinaciones de B y E:
P(A=1) = P(B=0, E=0, A=1)
Para que fuera más general, implementaremos la función que recorra todas las combinaciones posibles de B y E, y sume las probabilidades conjuntas correspondientes a A=1.

In [23]:
def inferencia_marginal(query, evidencia=None):
    if evidencia is None:
        evidencia = {}
    total = 0.0 # para normalizar
    for b in [0, 1]:
        for e in [0, 1]:
            for a in [0, 1]:
                estado = {'B': b, 'E': e, 'A': a}
                
                cumple_query = all(estado[var] == val for var, val in query.items())
                cumple_evidencia = all(estado[var] == val for var, val in evidencia.items())
                if cumple_query and cumple_evidencia:
                    total += prob_conjunta(b, e, a)
    return total

In [24]:
p_a1= inferencia_marginal(query={'A': 1})
print(f"P(A=1) = {p_a1}")

P(A=1) = 0.0199


## interpretación de resultados
Al calcular P(A=1) estamos obteniendo la probabilidad de que la alarma suene, considerando todas las posibles combinaciones de robo y terremoto. Dado que tanto el robo como el terremoto son eventos raros (con probabilidad 0.01), la probabilidad de que la alarma suene también será baja, pero no nula, debido a la naturaleza OR de la alarma.

## 5. Probabilidad condicional

Ahora, si para el task 2.3 queremos calcular las probabilidades condicionales. 
Usaremos la definición:
P(X | Y) = P(X, Y) / P(Y)
En este caso, si queremos calcular P(B=1 | A=1), necesitamos calcular P(B=1, A=1) y P(A=1).

In [25]:
def prob_condicional(query, evidencia):
    numerador = inferencia_marginal(query, evidencia)
    denominador = inferencia_marginal(evidencia)
    if denominador == 0:
        return 0
    return numerador / denominador

## 6. Task 2.3 - Demostración del efecto Explain Away 

para este último queremos calcular exactamente estas:

1. P( B = 1 | A = 1 )
2. P( B = 1 | A = 1, E = 1 )

Luego compararlos, 
- Si el primer caso solo sabemos que la alarma sonó
- En el segundo caso, sabemos que hubo un terremoto 

El terremoto explica la alarma, entonces la necesidad de explicar la alarma mediante un robo disminuye, por lo que P(B=1 | A=1, E=1) < P(B=1 | A=1). Esto es el efecto Explain Away.

In [26]:
# simple P( B = 1 | A = 1)
p_b_dado_a = prob_condicional(query={'B': 1}, evidencia={'A': 1})

# P Explain away P( B = 1 | A = 1, E = 1)
p_b_dado_a_e = prob_condicional(query={'B': 1}, evidencia={'A': 1, 'E': 1})

print(f"P(B=1 | A=1) = {p_b_dado_a:.4f}")
print(f"P(B=1 | A=1, E=1) = {p_b_dado_a_e:.4f}")

P(B=1 | A=1) = 0.5025
P(B=1 | A=1, E=1) = 0.0100


## Conclusiones 
podemos notar que la probabilidad de que haya habido un robo dado que la alarma sonó es baja, pero no nula. Sin embargo, si sabemos que hubo un terremoto, la probabilidad de que haya habido un robo dado que la alarma sonó disminuye aún más, lo que demuestra el efecto Explain Away en esta red bayesiana.

- Cuando solo sabemos que la alarma sonó, la probabilidad de robo aumenta considerablemente.
- Pero cuando además sabemos que hubo un terremoto, la probabilidad de robo disminuye, ya que el terremoto puede explicar la alarma sin necesidad de un robo.
- Por ello, la probabilidad de que también haya sido un robo se baja. 

Aunque el robo y el terremoto sean independientes a priori, la observación de la alarma crea una dependencia entre ellos, lo que es un ejemplo clásico del efecto Explain Away en redes bayesianas.